# 02 · Data preparation and leakage-safe splitting

Implements `PROJECT_PLAN.md` §3.2–3.3: separate the inputs from the label, build a reproducible partition, and fit preprocessing on permitted training data only. All 284,807 rows are used; nothing is dropped.

The same results can be produced without the notebook by running `python scripts/prepare_data.py`. Tests: `python tests/test_data_pipeline.py`.

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/genAI")  # change this if the project folder lives elsewhere on Drive
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.resolve()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT.name)

Project root: genAI


In [2]:
import numpy as np
import pandas as pd

from src.data.checks import run_leakage_checks
from src.data.load import load_raw, separate_features_target
from src.data.prepare import fit_transformers, load_experiment_inputs, prepare_data
from src.data.split import assign_groups, experiment_table, partition_table, stratified_group_split
from src.utils.io import load_config

pd.set_option("display.width", 200)
cfg = load_config(PROJECT_ROOT)
df = load_raw(PROJECT_ROOT, cfg)
print(f"Loaded {len(df):,} rows (SHA-256 verified)")

Loaded 284,807 rows (SHA-256 verified)


## 1. Separate input features from the `Class` target

In [3]:
X, y = separate_features_target(df)
print("Model-input columns (before transformation):", list(X.columns))
print("Class in X:", "Class" in X.columns, "| Time in X:", "Time" in X.columns)
print("Labels kept separately:", y.shape, "values", np.unique(y))

Model-input columns (before transformation): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
Class in X: False | Time in X: False
Labels kept separately: (284807,) values [0 1]


`Class` is never part of `X`. `Time` is also excluded from model inputs: it counts seconds since the first transaction, so it acts as a row index rather than a property of the transaction (see notebook 01, finding 4).

## 2. Group identical feature vectors

In [4]:
groups = assign_groups(df)
sizes = np.bincount(groups)
print(f"{groups.max() + 1:,} groups for {len(df):,} rows; {int((sizes > 1).sum()):,} groups have more than one row (largest: {sizes.max()})")

275,663 groups for 284,807 rows; 5,149 groups have more than one row (largest: 77)


Rows with identical `V1–V28 + Amount` values share a group, and a whole group always goes to one partition. That stops a duplicate of a training row from appearing in validation or test, which would make reconstruction look better than it is, without deleting any rows.

## 3. Stratified group split (70 / 15 / 15, seed 42)

In [5]:
splits = stratified_group_split(groups, y, cfg["split"]["ratios"], cfg["split"]["seed"])
partition_table(splits, y).round(4)

,rows,normal,fraud,fraud_pct,share_of_dataset_pct
split,,,,,
train,199366,199021,345,0.1730,70.0004
val,42721,42647,74,0.1732,15.0000
test,42720,42647,73,0.1709,14.9996
total,284807,284315,492,0.1727,100.0000


In [6]:
experiment_table(splits, y)

,rows,normal,fraud,used_for
partition,,,,
e1_train,199021,199021,0,E1 training (normal only)
e1_val_calibration,42647,42647,0,E1 early stopping + threshold calibration (nor...
e2_train,199366,199021,345,"E2 training (all train rows, labels ignored)"
e2_val,42721,42647,74,E2 early stopping + threshold calibration (all...
test,42720,42647,73,Final evaluation of E1 and E2 (labels read onc...
train_fraud,345,0,345,Excluded from E1 training; part of E2 training...
val_fraud,74,0,74,Not used for any development decision


**How each partition is used** (labels are used only to build these partitions and, at the very end, to evaluate):
- **E1 training (primary VAE):** normal training transactions only (199,021 rows). The 345 training frauds are left out.
- **E1 validation:** normal validation transactions only (42,647 rows), used for early stopping and threshold calibration. The 74 validation frauds are not used for any decision.
- **E2 training and validation (unsupervised VAE):** all training rows (199,366) and all validation rows (42,721), labels ignored.
- **Test (both experiments):** 42,720 unseen transactions (42,647 normal, 73 fraud). Test labels are read only once, after predictions are saved.

## 4. Fit preprocessing on permitted training rows only

In [7]:
transformers = fit_transformers(df, splits)
params = pd.DataFrame({
    "E1 mean": transformers["e1"].mean_, "E1 std": transformers["e1"].scale_,
    "E2 mean": transformers["e2"].mean_, "E2 std": transformers["e2"].scale_,
}, index=transformers["e1"].feature_names)
print("E1 fitted on:", transformers["e1"].fitted_on, f"({transformers['e1'].n_fit_rows:,} rows)")
print("E2 fitted on:", transformers["e2"].fitted_on, f"({transformers['e2'].n_fit_rows:,} rows)")
params.round(4)

E1 fitted on: e1_train (normal training rows) (199,021 rows)
E2 fitted on: e2_train (all training rows, labels ignored) (199,366 rows)


,E1 mean,E1 std,E2 mean,E2 std
V1,0.0107,1.9274,0.0021,1.9578
V2,-0.0053,1.6263,0.0011,1.6425
V3,0.0094,1.4636,-0.0032,1.5229
V4,-0.0080,1.3963,-0.0000,1.4133
V5,0.0071,1.3627,0.0014,1.3864
V6,0.0036,1.3327,0.0012,1.3351
V7,0.0104,1.1867,0.0004,1.2456
V8,-0.0015,1.1622,-0.0007,1.1989
V9,0.0029,1.0878,-0.0016,1.0970
V10,0.0097,1.0378,-0.0003,1.0831


The transform is `x -> (f(x) - mean) / std`, where `f` keeps V1–V28 as they are and replaces `Amount` with `log1p(Amount)`. The mean and population std come **only** from the experiment's training rows. Validation and test rows are transformed with those stored values and are never refitted. Because the V features have unequal variances (std 0.33–1.96), all 29 inputs are standardised.

## 5. Apply the fitted transformations consistently

In [8]:
e1 = transformers["e1"]
rows = {
    "E1 train (normal)": splits["train"][y[splits["train"]] == 0],
    "E1 validation (normal)": splits["val"][y[splits["val"]] == 0],
    "Test (all)": splits["test"],
}
summary = {}
for name, idx in rows.items():
    Z = e1.transform(df.iloc[idx])
    summary[name] = {"rows": len(idx), "mean of feature means": Z.mean(0).mean(), "max |feature mean|": np.abs(Z.mean(0)).max(),
                     "mean of feature stds": Z.std(0).mean()}
pd.DataFrame(summary).T.round(4)

,rows,mean of feature means,max |feature mean|,mean of feature stds
E1 train (normal),199021.0,-0.0000,0.0000,1.0000
E1 validation (normal),42647.0,-0.0003,0.0141,0.9957
Test (all),42720.0,-0.0024,0.0233,1.0296


The training rows are exactly standardised (feature means of 0 and stds of 1). Validation and test are close to, but not exactly, 0 and 1, which is expected when they are transformed with training statistics. The test row includes frauds, whose extreme values raise the average std.

## 6. Inverse transformation (needed for synthetic-data export)

In [9]:
sample = df.iloc[splits["test"][:5]][["V1", "V2", "V28", "Amount"]]
restored = e1.inverse_transform(e1.transform(df.iloc[splits["test"][:5]]))[["V1", "V2", "V28", "Amount"]]
display(pd.concat({"original": sample.reset_index(drop=True), "inverse(transform(x))": restored}, axis=1).round(4))
synthetic_like = e1.inverse_transform(np.random.default_rng(0).normal(size=(3, 29)))
synthetic_like.round(3)

original                         inverse(transform(x))                        
        V1      V2     V28  Amount                    V1      V2     V28  Amount
0  -1.3598 -0.0728 -0.0211  149.62               -1.3598 -0.0728 -0.0211  149.62
1  -1.3584 -1.3402 -0.0598  378.66               -1.3584 -1.3402 -0.0598  378.66
2   0.9964 -0.1226  0.0051   20.53                0.9964 -0.1226  0.0051   20.53
3  -0.6079  1.0313 -0.0330   14.80               -0.6079  1.0313 -0.0330   14.80
4  -0.2915  0.4456  0.0629   18.95               -0.2915  0.4456  0.0629   18.95

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,0.253,-0.220,0.947,0.138,-0.723,0.486,1.558,1.099,-0.763,-1.304,-0.630,0.050,-2.310,-0.182,-1.141,-0.610,-0.396,-0.259,0.335,0.799,-0.094,0.987,-0.408,0.214,0.469,0.046,-0.300,-0.300,9.97
1,0.435,-1.647,-0.297,-0.230,0.744,0.290,0.432,-0.761,-0.138,0.823,1.495,-1.182,1.508,1.219,0.714,0.230,-0.224,1.205,1.594,1.383,0.943,0.258,-0.741,-0.001,0.341,-0.621,0.157,0.140,72.99
2,-2.272,-1.081,-0.629,-1.641,2.377,-0.657,0.401,-0.302,1.725,1.380,0.631,-2.076,0.054,0.626,0.918,-0.513,1.377,-1.088,-0.537,0.717,0.034,1.447,0.115,-0.382,-0.197,-0.526,-0.514,0.205,60.18


`inverse_transform` maps model-space vectors back to `V1–V28` plus `Amount`: it undoes the standardisation, applies `expm1` to recover `Amount`, clips `Amount` at 0, and rounds it to 2 decimals. Generated samples are exported this way. They remain vectors in the **anonymised PCA feature space**, not interpretable transaction records.

## 7. Leakage and integrity checks

In [10]:
checks = pd.DataFrame(run_leakage_checks(df, splits, transformers, cfg))
assert checks["passed"].all(), checks[~checks["passed"]]
print(f"{int(checks['passed'].sum())}/{len(checks)} checks passed")
checks

17/17 checks passed


,check,passed,detail
0,splits_are_disjoint,True,"284,807 assigned indices, 284,807 unique"
1,every_row_assigned_no_subset,True,"284,807 of 284,807 rows assigned"
2,no_group_spans_two_splits,True,"0 of 275,663 groups span >1 split"
3,no_identical_vector_in_train_and_val_or_test,True,"identical-to-train rows: val 0, test 0 (indepe..."
4,class_rate_preserved_in_each_split,True,"fraud % train 0.1730, val 0.1732, test 0.1709 ..."
5,e1_training_set_is_normal_only,True,"199,021 rows, 0 fraud"
6,e1_calibration_set_is_normal_only,True,"42,647 rows, 0 fraud"
7,test_set_has_unseen_normal_and_fraud,True,"test: 42,647 normal, 73 fraud"
8,e1_model_inputs_exclude_class_and_time,True,"29 inputs: V1-V28, log1p_Amount"
9,e2_model_inputs_exclude_class_and_time,True,"29 inputs: V1-V28, log1p_Amount"


## 8. Save the partitions and fitted transformers

In [11]:
result = prepare_data(PROJECT_ROOT, cfg)  # re-runs the same deterministic steps, checks the raw-file hash, saves artefacts
processed = PROJECT_ROOT / cfg["paths"]["processed_dir"]
print("Saved to data/processed:", sorted(p.name for p in processed.iterdir()))
print("Saved to outputs/metrics:", ["leakage_checks.json", "partition_stats.json"])
print("Raw dataset unchanged:", [c for c in result["checks"] if c["check"] == "raw_dataset_unchanged"][0]["passed"])

Saved to data/processed: ['experiment_partitions.csv', 'split_summary.csv', 'splits.json', 'transformer_e1.json', 'transformer_e2.json']
Saved to outputs/metrics: ['leakage_checks.json', 'partition_stats.json']
Raw dataset unchanged: True


## 9. How later phases load the prepared data

In [12]:
data = load_experiment_inputs(PROJECT_ROOT, cfg, "e1")
{k: (v.shape, str(v.dtype)) for k, v in data.items() if isinstance(v, np.ndarray) and v.ndim == 2}

{'X_train': ((199021, 29), 'float32'), 'X_val': ((42647, 29), 'float32'), 'X_test': ((42720, 29), 'float32')}

`load_experiment_inputs` returns **features only**: no labels, and no `Time`. Test labels come from a separate function (`load_test_labels`) that phase 8 calls only after test predictions are written to disk.

## 10. Preprocessing decisions

| Item | Decision | Reason |
|---|---|---|
| Rows | All 284,807 kept | No subsetting; duplicates handled by grouping |
| Split | Stratified **group** split 70/15/15, seed 42 → 199,366 / 42,721 / 42,720 rows | Keeps the fraud rate (0.173 / 0.173 / 0.171%) and keeps duplicate vectors in one partition |
| Model inputs | V1–V28 + `log1p(Amount)` (29 features) | `Class` is the target; `Time` acts as a row index |
| `Amount` | `log1p`, then standardised | Skew 16.98 → 0.16 |
| `Time` | Excluded from inputs; kept in the raw data for EDA and a later hour-of-day sensitivity analysis | Not a transaction property; would be out of range under a time-ordered split |
| V1–V28 | Standardised (std ranged 0.33–1.96 before) | PCA components are centred but not unit-scaled |
| Scaler fit rows | E1: normal training rows; E2: all training rows | Validation and test never influence preprocessing |
| Saved objects | `data/processed/transformer_e1.json`, `transformer_e2.json` (plain JSON: means, stds, feature names) | Portable across Python/scikit-learn versions; supports `inverse_transform` |
| Saved partition | `data/processed/splits.json` with SHA-256 fingerprints of each index list | Reproducible; loading verifies the fingerprints |
| Extreme values | Kept | They carry the anomaly signal; robustness is handled in training (gradient clipping) and a `RobustScaler` sensitivity run |